# STEP 2 — Market Basket Analysis (FP-Growth), with parameter sweep  
====================================================================  
Reverted the per-NHS-product grouped approach (built to dodge an OOM crash  
on the full 13.8M-basket run) back to a single sample-based run, per Jack  
Denham's guidance (meeting notes, [date]):  
  
  "I would try 0.002, and play a bit until you find the sweet spot between  
  'runs in a reasonable amount of time' and 'results in a decent number of  
  rules'... Maybe the best way would be a parameter sweep with a timeout,  
  and then return time taken and number of rules returned. timeout = 300  
  seconds or something. start sparse like min_support = [0.0002, 0.002,  
  0.02] and narrow down based on the best results."  
  
Basket definition is UNCHANGED from the anchor-based design: one basket =  
one NHS (POM) purchase + every OTC product the same customer bought in the  
WINDOW_DAYS forward window after that NHS purchase.  
  
This version:  
  1. Samples SAMPLE_N_BASKETS baskets at random (default 500, per Jack:  
     "500 random basket is okay").  
  2. Sweeps MIN_SUPPORT_SWEEP x a set of sample sizes, each fpgrowth call  
     wrapped in a TIMEOUT_SECONDS timeout, reporting (time_taken,  
     num_frequent_itemsets, num_rules, num_otc_rules) per combination.  
  3. Lets you pick a final (sample_size, min_support) from the sweep table  
     and runs the real pipeline once at that setting, saving the result.  
  
Also directly addresses Jack's comment #1: results analysis should look at  
the most commonly recommended items and how dominant aspirin is — Part 7  
below reports exactly this (consequent frequency distribution) before  
saving.  
  
Requires: pip install mlxtend


**Requires `pip install mlxtend`** — run `%pip install mlxtend --quiet` first if not already installed, then restart the kernel.

In [7]:
import time
import signal
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

### CONFIG

In [12]:
WINDOW_DAYS = 30                  # forward-only window after each NHS anchor
MIN_CONFIDENCE = 0.3
OTC_LEGAL_CATS = {"GSL", "P"}

# Sweep settings (Jack's exact suggestion)
#MIN_SUPPORT_SWEEP = [0.0002, 0.002, 0.02]
#SAMPLE_SIZES_TO_SWEEP = [500, 5_000, 20_000]   # Jack: "play with sample size"
#TIMEOUT_SECONDS = 300

#MIN_SUPPORT_SWEEP = [0.0002, 0.0001, 0.00005]
#SAMPLE_SIZES_TO_SWEEP = [20_000, 50_000, 100_000]
#TIMEOUT_SECONDS = 600   # Jack: "300 or something" — not a hard rule, and
                         # bigger samples legitimately need more time

# Final run settings — set these AFTER reviewing the sweep table below
#FINAL_SAMPLE_SIZE = 500
#FINAL_MIN_SUPPORT = 0.002

FINAL_SAMPLE_SIZE = 50000    # was 500
FINAL_MIN_SUPPORT = 0.00008    # was 0.002


#MIN_SUPPORT_SWEEP = [0.01,0.02,0.05,0.001,0.002,0.005,0.0001,0.0002]        # 5 yox, 2 dəyər
#SAMPLE_SIZES_TO_SWEEP = [10_000, 20_000, 50_000, 100_000, 200_000, 500_000]  # 6 dəyər
TIMEOUT_SECONDS = 600

#MIN_COUNT_TARGET_OPTIONS = MIN_COUNT_TARGET_OPTIONS = list(range(2, 11))
#SAMPLE_SIZES_TO_SWEEP = [5_000, 10_000, 20_000, 50_000, 100_000, 200_000, 500_000]

MIN_COUNT_TARGET_OPTIONS = MIN_COUNT_TARGET_OPTIONS = [2]
SAMPLE_SIZES_TO_SWEEP = [5_000]


SAMPLE_SUPPORT_PAIRS = []
for sample_size in SAMPLE_SIZES_TO_SWEEP:
    for min_count in MIN_COUNT_TARGET_OPTIONS:
        SAMPLE_SUPPORT_PAIRS.append((sample_size, min_count / sample_size, min_count))

### STEP 1 — Load Step 1's outputs

In [9]:
transactions_df = pd.read_parquet("clean_transactions_all_drugs.parquet")
otc_products = pd.read_parquet("otc_product_universe.parquet")
cohort = pd.read_parquet("clean_cohort_all.parquet")

transactions_df["dispense_date_created"] = pd.to_datetime(transactions_df["dispense_date_created"])

print(f"[1] Loaded {len(transactions_df):,} transaction lines, "
      f"{transactions_df['CustomerKey'].nunique():,} customers")

[1] Loaded 67,124,000 transaction lines, 1,872,959 customers


### STEP 2 — Build NHS-anchor + forward-window-OTC baskets (unchanged design)

In [10]:
def build_nhs_otc_baskets(transactions_df, window_days=30):
    """
    One row per NHS anchor with >=1 OTC purchase in its forward window.
    Columns: CustomerKey, anchor_date, anchor_nhs_product, Items (list of
    [nhs_product] + matched OTC products). Forward-only window, matches
    the originally agreed spec (5 Sep NHS -> 5 Oct OTC included; 30 Nov
    OTC excluded).
    """
    nhs_df = transactions_df[transactions_df["LEGAL_CAT"] == "POM"][
        ["CustomerKey", "ProductKey", "dispense_date_created"]
    ].rename(columns={"ProductKey": "nhs_ProductKey", "dispense_date_created": "nhs_date"})
    nhs_df = nhs_df.reset_index(drop=True)
    nhs_df["anchor_id"] = nhs_df.index

    otc_df = transactions_df[transactions_df["LEGAL_CAT"].isin(OTC_LEGAL_CATS)][
        ["CustomerKey", "ProductKey", "dispense_date_created"]
    ].rename(columns={"ProductKey": "otc_ProductKey", "dispense_date_created": "otc_date"})

    merged = nhs_df.merge(otc_df, on="CustomerKey", how="inner")
    merged = merged[
        (merged["otc_date"] >= merged["nhs_date"])
        & (merged["otc_date"] <= merged["nhs_date"] + pd.Timedelta(days=window_days))
    ]

    matched_otc = merged.groupby("anchor_id")["otc_ProductKey"].apply(lambda s: list(set(s)))

    nhs_df = nhs_df[nhs_df["anchor_id"].isin(matched_otc.index)].copy()
    nhs_df["otc_products"] = nhs_df["anchor_id"].map(matched_otc)
    nhs_df["Items"] = nhs_df.apply(lambda r: [r["nhs_ProductKey"]] + r["otc_products"], axis=1)

    return nhs_df[["CustomerKey", "nhs_date", "nhs_ProductKey", "Items"]].rename(
        columns={"nhs_date": "anchor_date", "nhs_ProductKey": "anchor_nhs_product"}
    )

basket_df = build_nhs_otc_baskets(transactions_df, window_days=WINDOW_DAYS)
print(f"[2] {len(basket_df):,} NHS anchors have >=1 OTC purchase within "
      f"{WINDOW_DAYS} days forward (out of "
      f"{(transactions_df['LEGAL_CAT'] == 'POM').sum():,} total NHS dispense events)")

otc_keys = set(otc_products["ProductKey"].dropna())
nhs_keys = set(transactions_df.loc[transactions_df["LEGAL_CAT"] == "POM", "ProductKey"].dropna())

def all_otc(itemset):
    return all(item in otc_keys for item in itemset)

def all_nhs(itemset):
    return all(item in nhs_keys for item in itemset)

[2] 13,847,439 NHS anchors have >=1 OTC purchase within 30 days forward (out of 59,929,915 total NHS dispense events)


### STEP 3 — Timeout wrapper for fpgrowth (Jack's exact request)

In [11]:
class TimeoutException(Exception):
    pass

def _timeout_handler(signum, frame):
    raise TimeoutException()

def run_fpgrowth_with_timeout(matrix, min_support, timeout_seconds):
    """
    Unix-only (uses SIGALRM) — fine for AML Linux compute, would need a
    different mechanism (e.g. multiprocessing) on Windows.
    Returns (frequent_itemsets_or_None, elapsed_seconds, timed_out_bool).
    """
    signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(timeout_seconds)
    start = time.time()
    try:
        freq = fpgrowth(matrix, min_support=min_support, use_colnames=True)
        elapsed = time.time() - start
        signal.alarm(0)
        return freq, elapsed, False
    except TimeoutException:
        signal.alarm(0)
        return None, float(timeout_seconds), True

### STEP 4 — Parameter sweep (sample size x min_support), exactly per Jack

sample=20,000 support=0.0002: OK, 66.2s, itemsets=3027, rules=183, otc_rules=64, distinct_otc=8
sample=20,000 support=0.0001: OK, 311.6s, itemsets=10309, rules=55341, otc_rules=280, distinct_otc=61
sample=20,000 support=5e-05: TIMEOUT, 600.0s, itemsets=None, rules=None, otc_rules=None, distinct_otc=None
sample=50,000 support=0.0002: OK, 118.3s, itemsets=2436, rules=116, otc_rules=41, distinct_otc=2
sample=50,000 support=0.0001: OK, 360.7s, itemsets=5621, rules=324, otc_rules=87, distinct_otc=9

In [13]:
sweep_rows = []
matrix_cache = {}

for sample_size, min_sup, min_count in SAMPLE_SUPPORT_PAIRS:
    if sample_size not in matrix_cache:
        if len(basket_df) > sample_size:
            sample = basket_df.sample(n=sample_size, random_state=42)
        else:
            sample = basket_df
        baskets_sample = sample["Items"].tolist()
        te = TransactionEncoder()
        matrix = pd.DataFrame(te.fit(baskets_sample).transform(baskets_sample), columns=te.columns_)
        matrix_cache[sample_size] = matrix
    matrix = matrix_cache[sample_size]

    freq, elapsed, timed_out = run_fpgrowth_with_timeout(matrix, min_sup, TIMEOUT_SECONDS)
    if timed_out:
        n_itemsets, n_rules, n_otc_rules, n_distinct_otc = None, None, None, None
        status = "TIMEOUT"
    else:
        n_itemsets = len(freq)
        if n_itemsets == 0:
            n_rules, n_otc_rules, n_distinct_otc = 0, 0, 0
        else:
            rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
            n_rules = len(rules)
            if n_rules > 0:
                otc_mask = rules["antecedents"].apply(all_nhs) & rules["consequents"].apply(all_otc)
                n_otc_rules = otc_mask.sum()
                n_distinct_otc = rules.loc[otc_mask, "consequents"].apply(lambda s: tuple(sorted(s))).nunique()
            else:
                n_otc_rules, n_distinct_otc = 0, 0
        status = "OK"

    row = {
        "sample_size": sample_size, "min_count_target": min_count, "min_support": min_sup,
        "status": status, "elapsed_sec": round(elapsed, 1), "n_frequent_itemsets": n_itemsets,
        "n_raw_rules": n_rules, "n_otc_rules": n_otc_rules, "n_distinct_otc_products": n_distinct_otc,
    }
    sweep_rows.append(row)
    print(f"sample={sample_size:,} min_count={min_count} support={min_sup:.6f}: {status}, "
          f"{elapsed:.1f}s, itemsets={n_itemsets}, rules={n_rules}, "
          f"otc_rules={n_otc_rules}, distinct_otc={n_distinct_otc}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("fpgrowth_parameter_sweep.csv", index=False)
print("\n=== Sweep summary (saved to fpgrowth_parameter_sweep.csv) ===")
print(sweep_df.to_string(index=False))
print(
    "\nReview this table, then set FINAL_SAMPLE_SIZE / FINAL_MIN_SUPPORT "
    "in the CONFIG section above to the combination that best balances "
    "runtime against rule count/diversity, and re-run from Step 5 onward."
)

sample=5,000 min_count=2 support=0.000400: OK, 11.5s, itemsets=2195, rules=361, otc_rules=88, distinct_otc=22

=== Sweep summary (saved to fpgrowth_parameter_sweep.csv) ===
 sample_size  min_count_target  min_support status  elapsed_sec  n_frequent_itemsets  n_raw_rules  n_otc_rules  n_distinct_otc_products
        5000                 2       0.0004     OK         11.5                 2195          361           88                       22

Review this table, then set FINAL_SAMPLE_SIZE / FINAL_MIN_SUPPORT in the CONFIG section above to the combination that best balances runtime against rule count/diversity, and re-run from Step 5 onward.


In [6]:
sweep_rows = []
matrix_cache = {}

for sample_size, min_sup, min_count in SAMPLE_SUPPORT_PAIRS:
    if sample_size not in matrix_cache:
        if len(basket_df) > sample_size:
            sample = basket_df.sample(n=sample_size, random_state=42)
        else:
            sample = basket_df
        baskets_sample = sample["Items"].tolist()
        te = TransactionEncoder()
        matrix = pd.DataFrame(te.fit(baskets_sample).transform(baskets_sample), columns=te.columns_)
        matrix_cache[sample_size] = matrix
    matrix = matrix_cache[sample_size]

    freq, elapsed, timed_out = run_fpgrowth_with_timeout(matrix, min_sup, TIMEOUT_SECONDS)
    if timed_out:
        n_itemsets, n_rules, n_otc_rules, n_distinct_otc = None, None, None, None
        status = "TIMEOUT"
    else:
        n_itemsets = len(freq)
        if n_itemsets == 0:
            n_rules, n_otc_rules, n_distinct_otc = 0, 0, 0
        else:
            rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
            n_rules = len(rules)
            if n_rules > 0:
                otc_mask = rules["antecedents"].apply(all_nhs) & rules["consequents"].apply(all_otc)
                n_otc_rules = otc_mask.sum()
                n_distinct_otc = rules.loc[otc_mask, "consequents"].apply(lambda s: tuple(sorted(s))).nunique()
            else:
                n_otc_rules, n_distinct_otc = 0, 0
        status = "OK"

    row = {
        "sample_size": sample_size, "min_count_target": min_count, "min_support": min_sup,
        "status": status, "elapsed_sec": round(elapsed, 1), "n_frequent_itemsets": n_itemsets,
        "n_raw_rules": n_rules, "n_otc_rules": n_otc_rules, "n_distinct_otc_products": n_distinct_otc,
    }
    sweep_rows.append(row)
    print(f"sample={sample_size:,} min_count={min_count} support={min_sup:.6f}: {status}, "
          f"{elapsed:.1f}s, itemsets={n_itemsets}, rules={n_rules}, "
          f"otc_rules={n_otc_rules}, distinct_otc={n_distinct_otc}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("fpgrowth_parameter_sweep.csv", index=False)
print("\n=== Sweep summary (saved to fpgrowth_parameter_sweep.csv) ===")
print(sweep_df.to_string(index=False))
print(
    "\nReview this table, then set FINAL_SAMPLE_SIZE / FINAL_MIN_SUPPORT "
    "in the CONFIG section above to the combination that best balances "
    "runtime against rule count/diversity, and re-run from Step 5 onward."
)

sample=5,000 min_count=2 support=0.000400: OK, 11.6s, itemsets=2195, rules=361, otc_rules=88, distinct_otc=22
sample=5,000 min_count=3 support=0.000600: OK, 4.1s, itemsets=1081, rules=79, otc_rules=33, distinct_otc=4
sample=5,000 min_count=4 support=0.000800: OK, 2.2s, itemsets=739, rules=38, otc_rules=18, distinct_otc=1
sample=5,000 min_count=5 support=0.001000: OK, 1.4s, itemsets=552, rules=27, otc_rules=12, distinct_otc=1
sample=5,000 min_count=6 support=0.001200: OK, 0.9s, itemsets=439, rules=24, otc_rules=10, distinct_otc=1
sample=5,000 min_count=7 support=0.001400: OK, 0.8s, itemsets=373, rules=23, otc_rules=10, distinct_otc=1
sample=5,000 min_count=8 support=0.001600: OK, 0.6s, itemsets=331, rules=21, otc_rules=10, distinct_otc=1
sample=5,000 min_count=9 support=0.001800: OK, 0.5s, itemsets=299, rules=21, otc_rules=10, distinct_otc=1
sample=5,000 min_count=10 support=0.002000: OK, 0.5s, itemsets=281, rules=20, otc_rules=9, distinct_otc=1
sample=10,000 min_count=2 support=0.00020

In [6]:
sweep_rows = []
matrix_cache = {}  # avoid rebuilding the same sample's matrix repeatedly

for sample_size in SAMPLE_SIZES_TO_SWEEP:
    if sample_size not in matrix_cache:
        if len(basket_df) > sample_size:
            sample = basket_df.sample(n=sample_size, random_state=42)
        else:
            sample = basket_df
        baskets_sample = sample["Items"].tolist()
        te = TransactionEncoder()
        matrix = pd.DataFrame(te.fit(baskets_sample).transform(baskets_sample), columns=te.columns_)
        matrix_cache[sample_size] = matrix
    matrix = matrix_cache[sample_size]

    for min_sup in MIN_SUPPORT_SWEEP:
        freq, elapsed, timed_out = run_fpgrowth_with_timeout(matrix, min_sup, TIMEOUT_SECONDS)
        if timed_out:
            n_itemsets, n_rules, n_otc_rules, n_distinct_otc = None, None, None, None
            status = "TIMEOUT"
        else:
            n_itemsets = len(freq)
            if n_itemsets == 0:
                n_rules, n_otc_rules, n_distinct_otc = 0, 0, 0
            else:
                rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
                n_rules = len(rules)
                if n_rules > 0:
                    otc_mask = rules["antecedents"].apply(all_nhs) & rules["consequents"].apply(all_otc)
                    n_otc_rules = otc_mask.sum()
                    n_distinct_otc = rules.loc[otc_mask, "consequents"].apply(lambda s: tuple(sorted(s))).nunique()
                else:
                    n_otc_rules, n_distinct_otc = 0, 0
            status = "OK"

        row = {
            "sample_size": sample_size, "min_support": min_sup, "status": status,
            "elapsed_sec": round(elapsed, 1), "n_frequent_itemsets": n_itemsets,
            "n_raw_rules": n_rules, "n_otc_rules": n_otc_rules,
            "n_distinct_otc_products": n_distinct_otc,
        }
        sweep_rows.append(row)
        print(f"sample={sample_size:,} support={min_sup}: {status}, "
              f"{elapsed:.1f}s, itemsets={n_itemsets}, rules={n_rules}, "
              f"otc_rules={n_otc_rules}, distinct_otc={n_distinct_otc}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("fpgrowth_parameter_sweep.csv", index=False)
print("\n=== Sweep summary (saved to fpgrowth_parameter_sweep.csv) ===")
print(sweep_df.to_string(index=False))
print(
    "\nReview this table, then set FINAL_SAMPLE_SIZE / FINAL_MIN_SUPPORT "
    "in the CONFIG section above to the combination that best balances "
    "runtime against rule count/diversity, and re-run from Step 5 onward."
)

sample=10,000 support=0.01: OK, 0.3s, itemsets=47, rules=0, otc_rules=0, distinct_otc=0
sample=10,000 support=0.02: OK, 0.2s, itemsets=16, rules=0, otc_rules=0, distinct_otc=0
sample=10,000 support=0.05: OK, 0.2s, itemsets=2, rules=0, otc_rules=0, distinct_otc=0
sample=10,000 support=0.001: OK, 2.3s, itemsets=485, rules=28, otc_rules=12, distinct_otc=1
sample=10,000 support=0.002: OK, 1.0s, itemsets=260, rules=18, otc_rules=9, distinct_otc=1
sample=10,000 support=0.005: OK, 0.4s, itemsets=112, rules=3, otc_rules=1, distinct_otc=1
sample=10,000 support=0.0001: TIMEOUT, 600.0s, itemsets=None, rules=None, otc_rules=None, distinct_otc=None
sample=10,000 support=0.0002: OK, 56.8s, itemsets=4446, rules=943, otc_rules=163, distinct_otc=35
sample=20,000 support=0.01: OK, 0.6s, itemsets=48, rules=0, otc_rules=0, distinct_otc=0
sample=20,000 support=0.02: OK, 0.6s, itemsets=15, rules=0, otc_rules=0, distinct_otc=0
sample=20,000 support=0.05: OK, 0.6s, itemsets=2, rules=0, otc_rules=0, distinct_o

In [9]:
MIN_SUPPORT_SWEEP = [0.0001,0.0002,0.0005,0.00001,0.00002,0.00005]        # 5 yox, 2 dəyər
SAMPLE_SIZES_TO_SWEEP = [500,1000,5000,10_000, 20_000, 50_000, 100_000, 200_000, 500_000,1_000_000]  # 6 dəyər
TIMEOUT_SECONDS = 600

In [10]:
sweep_rows = []
matrix_cache = {}  # avoid rebuilding the same sample's matrix repeatedly

for sample_size in SAMPLE_SIZES_TO_SWEEP:
    if sample_size not in matrix_cache:
        if len(basket_df) > sample_size:
            sample = basket_df.sample(n=sample_size, random_state=42)
        else:
            sample = basket_df
        baskets_sample = sample["Items"].tolist()
        te = TransactionEncoder()
        matrix = pd.DataFrame(te.fit(baskets_sample).transform(baskets_sample), columns=te.columns_)
        matrix_cache[sample_size] = matrix
    matrix = matrix_cache[sample_size]

    for min_sup in MIN_SUPPORT_SWEEP:
        freq, elapsed, timed_out = run_fpgrowth_with_timeout(matrix, min_sup, TIMEOUT_SECONDS)
        if timed_out:
            n_itemsets, n_rules, n_otc_rules, n_distinct_otc = None, None, None, None
            status = "TIMEOUT"
        else:
            n_itemsets = len(freq)
            if n_itemsets == 0:
                n_rules, n_otc_rules, n_distinct_otc = 0, 0, 0
            else:
                rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
                n_rules = len(rules)
                if n_rules > 0:
                    otc_mask = rules["antecedents"].apply(all_nhs) & rules["consequents"].apply(all_otc)
                    n_otc_rules = otc_mask.sum()
                    n_distinct_otc = rules.loc[otc_mask, "consequents"].apply(lambda s: tuple(sorted(s))).nunique()
                else:
                    n_otc_rules, n_distinct_otc = 0, 0
            status = "OK"

        row = {
            "sample_size": sample_size, "min_support": min_sup, "status": status,
            "elapsed_sec": round(elapsed, 1), "n_frequent_itemsets": n_itemsets,
            "n_raw_rules": n_rules, "n_otc_rules": n_otc_rules,
            "n_distinct_otc_products": n_distinct_otc,
        }
        sweep_rows.append(row)
        print(f"sample={sample_size:,} support={min_sup}: {status}, "
              f"{elapsed:.1f}s, itemsets={n_itemsets}, rules={n_rules}, "
              f"otc_rules={n_otc_rules}, distinct_otc={n_distinct_otc}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("fpgrowth_parameter_sweep.csv", index=False)
print("\n=== Sweep summary (saved to fpgrowth_parameter_sweep.csv) ===")
print(sweep_df.to_string(index=False))
print(
    "\nReview this table, then set FINAL_SAMPLE_SIZE / FINAL_MIN_SUPPORT "
    "in the CONFIG section above to the combination that best balances "
    "runtime against rule count/diversity, and re-run from Step 5 onward."
)

sample=500 support=0.0001: OK, 7.6s, itemsets=7775, rules=580262, otc_rules=3374, distinct_otc=3044
sample=500 support=0.0002: OK, 7.7s, itemsets=7775, rules=580262, otc_rules=3374, distinct_otc=3044
sample=500 support=0.0005: OK, 7.7s, itemsets=7775, rules=580262, otc_rules=3374, distinct_otc=3044
sample=500 support=1e-05: OK, 7.6s, itemsets=7775, rules=580262, otc_rules=3374, distinct_otc=3044
sample=500 support=2e-05: OK, 7.6s, itemsets=7775, rules=580262, otc_rules=3374, distinct_otc=3044
sample=500 support=5e-05: OK, 7.7s, itemsets=7775, rules=580262, otc_rules=3374, distinct_otc=3044
sample=1,000 support=0.0001: OK, 18.6s, itemsets=9883, rules=584421, otc_rules=3933, distinct_otc=3333
sample=1,000 support=0.0002: OK, 18.7s, itemsets=9883, rules=584421, otc_rules=3933, distinct_otc=3333
sample=1,000 support=0.0005: OK, 18.6s, itemsets=9883, rules=584421, otc_rules=3933, distinct_otc=3333
sample=1,000 support=1e-05: OK, 18.8s, itemsets=9883, rules=584421, otc_rules=3933, distinct_o

In [ ]:
MIN_SUPPORT_SWEEP = [0.01,0.02,0.05,0.001,0.002,0.005,0.0001,0.0002]        # 5 yox, 2 dəyər
SAMPLE_SIZES_TO_SWEEP = [500,1000,5000]  # 6 dəyər
TIMEOUT_SECONDS = 600

In [ ]:
sweep_rows = []
matrix_cache = {}  # avoid rebuilding the same sample's matrix repeatedly

for sample_size in SAMPLE_SIZES_TO_SWEEP:
    if sample_size not in matrix_cache:
        if len(basket_df) > sample_size:
            sample = basket_df.sample(n=sample_size, random_state=42)
        else:
            sample = basket_df
        baskets_sample = sample["Items"].tolist()
        te = TransactionEncoder()
        matrix = pd.DataFrame(te.fit(baskets_sample).transform(baskets_sample), columns=te.columns_)
        matrix_cache[sample_size] = matrix
    matrix = matrix_cache[sample_size]

    for min_sup in MIN_SUPPORT_SWEEP:
        freq, elapsed, timed_out = run_fpgrowth_with_timeout(matrix, min_sup, TIMEOUT_SECONDS)
        if timed_out:
            n_itemsets, n_rules, n_otc_rules, n_distinct_otc = None, None, None, None
            status = "TIMEOUT"
        else:
            n_itemsets = len(freq)
            if n_itemsets == 0:
                n_rules, n_otc_rules, n_distinct_otc = 0, 0, 0
            else:
                rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
                n_rules = len(rules)
                if n_rules > 0:
                    otc_mask = rules["antecedents"].apply(all_nhs) & rules["consequents"].apply(all_otc)
                    n_otc_rules = otc_mask.sum()
                    n_distinct_otc = rules.loc[otc_mask, "consequents"].apply(lambda s: tuple(sorted(s))).nunique()
                else:
                    n_otc_rules, n_distinct_otc = 0, 0
            status = "OK"

        row = {
            "sample_size": sample_size, "min_support": min_sup, "status": status,
            "elapsed_sec": round(elapsed, 1), "n_frequent_itemsets": n_itemsets,
            "n_raw_rules": n_rules, "n_otc_rules": n_otc_rules,
            "n_distinct_otc_products": n_distinct_otc,
        }
        sweep_rows.append(row)
        print(f"sample={sample_size:,} support={min_sup}: {status}, "
              f"{elapsed:.1f}s, itemsets={n_itemsets}, rules={n_rules}, "
              f"otc_rules={n_otc_rules}, distinct_otc={n_distinct_otc}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("fpgrowth_parameter_sweep.csv", index=False)
print("\n=== Sweep summary (saved to fpgrowth_parameter_sweep.csv) ===")
print(sweep_df.to_string(index=False))
print(
    "\nReview this table, then set FINAL_SAMPLE_SIZE / FINAL_MIN_SUPPORT "
    "in the CONFIG section above to the combination that best balances "
    "runtime against rule count/diversity, and re-run from Step 5 onward."
)

In [ ]:
MIN_SUPPORT_SWEEP = [0.01,0.02,0.05,0.001,0.002,0.005,0.0001,0.0002,0.0005,0.00001,0.00002,0.00005]        # 5 yox, 2 dəyər
SAMPLE_SIZES_TO_SWEEP = [2000,3000,4000,6000,7000,8000,9000]  # 6 dəyər
TIMEOUT_SECONDS = 600



In [30]:
MIN_SUPPORT_SWEEP = [0.01,0.02,0.002,0.005,0.0001,0.0002]        # 5 yox, 2 dəyər
SAMPLE_SIZES_TO_SWEEP = [2000,3000,4000,6000,7000,8000,9000]  # 6 dəyər
TIMEOUT_SECONDS = 600

In [31]:
sweep_rows = []
matrix_cache = {}  # avoid rebuilding the same sample's matrix repeatedly

for sample_size in SAMPLE_SIZES_TO_SWEEP:
    if sample_size not in matrix_cache:
        if len(basket_df) > sample_size:
            sample = basket_df.sample(n=sample_size, random_state=42)
        else:
            sample = basket_df
        baskets_sample = sample["Items"].tolist()
        te = TransactionEncoder()
        matrix = pd.DataFrame(te.fit(baskets_sample).transform(baskets_sample), columns=te.columns_)
        matrix_cache[sample_size] = matrix
    matrix = matrix_cache[sample_size]

    for min_sup in MIN_SUPPORT_SWEEP:
        freq, elapsed, timed_out = run_fpgrowth_with_timeout(matrix, min_sup, TIMEOUT_SECONDS)
        if timed_out:
            n_itemsets, n_rules, n_otc_rules, n_distinct_otc = None, None, None, None
            status = "TIMEOUT"
        else:
            n_itemsets = len(freq)
            if n_itemsets == 0:
                n_rules, n_otc_rules, n_distinct_otc = 0, 0, 0
            else:
                rules = association_rules(freq, metric="confidence", min_threshold=MIN_CONFIDENCE)
                n_rules = len(rules)
                if n_rules > 0:
                    otc_mask = rules["antecedents"].apply(all_nhs) & rules["consequents"].apply(all_otc)
                    n_otc_rules = otc_mask.sum()
                    n_distinct_otc = rules.loc[otc_mask, "consequents"].apply(lambda s: tuple(sorted(s))).nunique()
                else:
                    n_otc_rules, n_distinct_otc = 0, 0
            status = "OK"

        row = {
            "sample_size": sample_size, "min_support": min_sup, "status": status,
            "elapsed_sec": round(elapsed, 1), "n_frequent_itemsets": n_itemsets,
            "n_raw_rules": n_rules, "n_otc_rules": n_otc_rules,
            "n_distinct_otc_products": n_distinct_otc,
        }
        sweep_rows.append(row)
        print(f"sample={sample_size:,} support={min_sup}: {status}, "
              f"{elapsed:.1f}s, itemsets={n_itemsets}, rules={n_rules}, "
              f"otc_rules={n_otc_rules}, distinct_otc={n_distinct_otc}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("fpgrowth_parameter_sweep.csv", index=False)
print("\n=== Sweep summary (saved to fpgrowth_parameter_sweep.csv) ===")
print(sweep_df.to_string(index=False))
print(
    "\nReview this table, then set FINAL_SAMPLE_SIZE / FINAL_MIN_SUPPORT "
    "in the CONFIG section above to the combination that best balances "
    "runtime against rule count/diversity, and re-run from Step 5 onward."
)

sample=2,000 support=0.01: OK, 0.1s, itemsets=47, rules=1, otc_rules=0, distinct_otc=0
sample=2,000 support=0.02: OK, 0.0s, itemsets=18, rules=0, otc_rules=0, distinct_otc=0
sample=2,000 support=0.002: OK, 0.2s, itemsets=323, rules=27, otc_rules=9, distinct_otc=1
sample=2,000 support=0.005: OK, 0.1s, itemsets=128, rules=7, otc_rules=5, distinct_otc=1
sample=2,000 support=0.0001: OK, 60.8s, itemsets=16159, rules=680546, otc_rules=5334, distinct_otc=4242
sample=2,000 support=0.0002: OK, 60.2s, itemsets=16159, rules=680546, otc_rules=5334, distinct_otc=4242
sample=3,000 support=0.01: OK, 0.0s, itemsets=47, rules=0, otc_rules=0, distinct_otc=0
sample=3,000 support=0.02: OK, 0.0s, itemsets=17, rules=0, otc_rules=0, distinct_otc=0
sample=3,000 support=0.002: OK, 0.3s, itemsets=303, rules=25, otc_rules=11, distinct_otc=1
sample=3,000 support=0.005: OK, 0.1s, itemsets=113, rules=6, otc_rules=4, distinct_otc=1
sample=3,000 support=0.0001: OK, 122.4s, itemsets=21602, rules=776128, otc_rules=6251

### STEP 5 — Final run at the chosen (FINAL_SAMPLE_SIZE, FINAL_MIN_SUPPORT)

In [14]:
if FINAL_SAMPLE_SIZE in matrix_cache:
    final_matrix = matrix_cache[FINAL_SAMPLE_SIZE]
else:
    if len(basket_df) > FINAL_SAMPLE_SIZE:
        final_sample = basket_df.sample(n=FINAL_SAMPLE_SIZE, random_state=42)
    else:
        final_sample = basket_df
    final_baskets = final_sample["Items"].tolist()
    te_final = TransactionEncoder()
    final_matrix = pd.DataFrame(
        te_final.fit(final_baskets).transform(final_baskets), columns=te_final.columns_
    )

print(f"\n[5] Final run: sample_size={FINAL_SAMPLE_SIZE:,}, min_support={FINAL_MIN_SUPPORT}")
frequent_itemsets = fpgrowth(final_matrix, min_support=FINAL_MIN_SUPPORT, use_colnames=True)
print(f"[5] {len(frequent_itemsets):,} frequent itemsets")

if len(frequent_itemsets) == 0:
    raise SystemExit(
        "No frequent itemsets at this FINAL_MIN_SUPPORT — pick a lower "
        "value from the sweep table above."
    )

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules.sort_values(["support", "lift"], ascending=False)
print(f"[5] {len(rules):,} raw association rules")

rules["antecedent_all_nhs"] = rules["antecedents"].apply(all_nhs)
rules["consequent_all_otc"] = rules["consequents"].apply(all_otc)
otc_rules = rules[rules["antecedent_all_nhs"] & rules["consequent_all_otc"]].copy()
print(f"[5] {len(otc_rules):,} rules with an all-NHS antecedent AND all-OTC consequent")


[5] Final run: sample_size=50,000, min_support=8e-05


### STEP 6 — Labels

In [ ]:
name_map = transactions_df.drop_duplicates("ProductKey").set_index("ProductKey")["DrugName"]

def label(itemset):
    return ", ".join(str(name_map.get(i, i)) for i in itemset)

otc_rules["antecedent_names"] = otc_rules["antecedents"].apply(label)
otc_rules["consequent_names"] = otc_rules["consequents"].apply(label)
otc_rules = otc_rules.sort_values(["support", "lift"], ascending=False)

### STEP 7 — Jack's comment #1: how dominant is the top recommended item?

In [ ]:
consequent_freq = otc_rules["consequent_names"].value_counts()
print("\n=== Consequent product frequency across surviving rules ===")
print(consequent_freq.to_string())
print(
    f"\nTop item ('{consequent_freq.index[0]}') accounts for "
    f"{consequent_freq.iloc[0]} / {len(otc_rules)} rules "
    f"({consequent_freq.iloc[0] / len(otc_rules):.1%}) — report this "
    "dominance figure directly in the results analysis, per Jack's note."
)

### STEP 8 — Save

In [ ]:
otc_rules["antecedents"] = otc_rules["antecedents"].apply(lambda s: tuple(sorted(s)))
otc_rules["consequents"] = otc_rules["consequents"].apply(lambda s: tuple(sorted(s)))

otc_rules.to_parquet("market_basket_otc_rules.parquet", index=False)
otc_rules.to_csv("market_basket_otc_rules.csv", index=False)

print(f"\nSaved {len(otc_rules):,} rules -> market_basket_otc_rules.parquet / .csv")
print("\nTop 10 NHS-to-OTC cross-sell rules by support, then lift:")
print(
    otc_rules[["antecedent_names", "consequent_names", "support", "confidence", "lift"]]
    .head(10)
    .to_string(index=False)
)